# Grover's Search — Amazon Braket

Grover's algorithm finds a marked item in an unsorted list of $N$
items using only $O(\sqrt{N})$ oracle queries.  On 3 qubits we
search $N = 8$ states and mark $|101\rangle$ (5).

In [ ]:
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## Oracle and diffusion operator

In [ ]:
def oracle_mark_101():
    """Phase flip on |101>."""
    c = Circuit()
    c.x(0)
    c.x(2)
    c.cz(0, 1)
    c.cz(1, 2)
    c.cz(0, 1)
    c.x(0)
    c.x(2)
    return c

def diffusion_operator(n=3):
    """Grover diffusion: 2|s><s| - I."""
    c = Circuit()
    for i in range(n):
        c.h(i)
    for i in range(n):
        c.x(i)
    c.h(n - 1)
    c.cz(0, 1)
    c.h(n - 1)
    for i in range(n):
        c.x(i)
    for i in range(n):
        c.h(i)
    return c

def grover_circuit(iterations=1, n=3):
    """Build Grover search circuit."""
    c = Circuit()
    for i in range(n):
        c.h(i)
    for _ in range(iterations):
        c.add_circuit(oracle_mark_101())
        c.add_circuit(diffusion_operator(n))
    for i in range(n):
        c.measure(i)
    return c

## 1 Grover iteration (optimal for N=8)

The optimal number of iterations is $\lfloor \pi/4 \sqrt{N} \rfloor = 2$
for $N=8$, but 1 iteration already gives $\sim 95\%$ probability.

In [ ]:
circuit = grover_circuit(iterations=1)
print(circuit)

result = device.run(circuit, shots=1000).result()
counts = result.result_types[0].value

print(f"counts: {counts}")
target_prob = counts.get('101', 0) / 1000
print(f"P(|101>) = {target_prob:.1%}  (expected ~94.5%)")

## 2 iterations (over-rotation)

Too many iterations decreases the probability — a characteristic
of Grover's algorithm.

In [ ]:
circuit = grover_circuit(iterations=2)

result = device.run(circuit, shots=1000).result()
counts = result.result_types[0].value

print(f"counts: {counts}")
target_prob = counts.get('101', 0) / 1000
print(f"P(|101>) = {target_prob:.1%}  (over-rotated, probability decreased)")

## Amplitude evolution

Inspect the amplitudes after each stage: uniform superposition,
oracle, and diffusion.

In [ ]:
n = 3

circuit = Circuit()
for i in range(n):
    circuit.h(i)
result = device.run(circuit, shots=0).result()
amps_0 = result.result_types[0].value
print("After H^n (uniform superposition):")
for k in range(8):
    bits = format(k, '03b')
    print(f"  |{bits}>  amp = {amps_0[k]:+.4f}  |amp|^2 = {abs(amps_0[k])**2:.4f}")
print()

circuit.add_circuit(oracle_mark_101())
result = device.run(circuit, shots=0).result()
amps_1 = result.result_types[0].value
print("After oracle (|101> phase flipped):")
for k in range(8):
    bits = format(k, '03b')
    print(f"  |{bits}>  amp = {amps_1[k]:+.4f}  |amp|^2 = {abs(amps_1[k])**2:.4f}")
print()

circuit.add_circuit(diffusion_operator(n))
result = device.run(circuit, shots=0).result()
amps_2 = result.result_types[0].value
print("After diffusion (amplitude amplified):")
for k in range(8):
    bits = format(k, '03b')
    marker = " <-- target" if k == 5 else ""
    print(f"  |{bits}>  amp = {amps_2[k]:+.4f}  |amp|^2 = {abs(amps_2[k])**2:.4f}{marker}")